In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import zipfile
import os

zip_path = '/content/drive/MyDrive/dev/GeoguessrAI/stitchedAllImages.zip'
extract_path = '/content/data/images'

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ Unzipped to:", extract_path)

✅ Unzipped to: /content/data/images


In [ ]:
import os
import pandas as pd

csv_path = 'data/labels.csv'
images_dir = 'data/images/stitchedAllImages'

df = pd.read_csv(csv_path)
print(df.head())
print(df.columns)


# Filter rows where image file exists
df['image_path'] = df['filename'].apply(lambda x: os.path.join(images_dir, x + '.jpg'))  # or '.png'
df = df[df['image_path'].apply(os.path.exists)].reset_index(drop=True)
print(df.head())



                   filename                                  labels
0  XiAoXAo3n32F-zlA81PCgQ -  39.7265625_40.078125_19.6875_20.390625
1  _Z4metv4Hr5aOWkhKDk4FA -      42.1875_42.890625_19.6875_21.09375
2  qYp3zps-LVzjxXoAA-ZF4g -  41.484375_41.8359375_19.6875_20.390625
3  jfQqkBSdJFuJaIUzeBL5yw -  41.1328125_41.484375_19.6875_20.390625
4  hEEptpO5B4G-_wKdr6KYOQ -  41.1328125_41.484375_19.6875_20.390625
Index(['filename', 'labels'], dtype='object')
                   filename                                  labels  \
0  XiAoXAo3n32F-zlA81PCgQ -  39.7265625_40.078125_19.6875_20.390625   
1  _Z4metv4Hr5aOWkhKDk4FA -      42.1875_42.890625_19.6875_21.09375   
2  qYp3zps-LVzjxXoAA-ZF4g -  41.484375_41.8359375_19.6875_20.390625   
3  jfQqkBSdJFuJaIUzeBL5yw -  41.1328125_41.484375_19.6875_20.390625   
4  hEEptpO5B4G-_wKdr6KYOQ -  41.1328125_41.484375_19.6875_20.390625   

                                          image_path  
0  data/images/stitchedAllImages/XiAoXAo3n32F-zlA...  
1  data/i

In [ ]:
#download filtered csv
filtered_csv_path = 'filtered_labels.csv'
df.to_csv(filtered_csv_path, index=False)
print(f"Filtered CSV saved to: {filtered_csv_path}")

Filtered CSV saved to: filtered_labels.csv


In [ ]:
from torch.utils.data import Dataset
from PIL import Image
import torchvision.transforms as transforms
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader, random_split

# 1. Encode labels to integers
le = LabelEncoder()
df['label_id'] = le.fit_transform(df['labels'])
num_classes = len(le.classes_)
print("Number of classes:", num_classes)


# 2. Dataset example
class CustomImageDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.loc[idx, 'image_path']
        label = self.df.loc[idx, 'label_id']  # integer label

        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)

        return image, label



Number of classes: 3855


In [ ]:
from torch.utils.data import DataLoader

img_height=224
img_width=224

print("Label min:", df['label_id'].min())
print("Label max:", df['label_id'].max())
print("Unique labels:", len(df['label_id'].unique()))
print("Number of classes:", num_classes)

#resize and normalize images
transform = transforms.Compose([
    transforms.Resize((img_width, img_height)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

dataset = CustomImageDataset(df, transform=transform)



Label min: 0
Label max: 3854
Unique labels: 3855
Number of classes: 3855


In [ ]:
# training/ validation split
train_size = int(0.9 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=4)


In [ ]:
import timm
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from tqdm import tqdm
import os
from torchmetrics.classification import MulticlassAccuracy, MulticlassPrecision, MulticlassRecall, MulticlassF1Score


num_epochs=50
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

backbone = timm.create_model('tiny_vit_11m_224', pretrained=True, num_classes=0)

#custom head on top of tinyvit backbone
class CustomTinyViT(nn.Module):
    def __init__(self, backbone, num_classes):
        super().__init__()
        self.backbone = backbone
        in_features = backbone.num_features
        self.classifier = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.GELU(),
            nn.Dropout(0.5),
            nn.Linear(512, 256),
            nn.GELU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.backbone(x)
        x = self.classifier(x)
        return x

model = CustomTinyViT(backbone, num_classes).to(device)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.03)

# Scheduler (with warmup + cosine)
import math
from torch.optim.lr_scheduler import LambdaLR
def cosine_schedule_with_warmup(optimizer, warmup_epochs, total_epochs, final_lr_factor=0.01):
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return (epoch + 1) / warmup_epochs
        else:
            progress = (epoch - warmup_epochs) / (total_epochs - warmup_epochs)
            return final_lr_factor + 0.5 * (1 - final_lr_factor) * (1 + math.cos(math.pi * progress))
    return LambdaLR(optimizer, lr_lambda=lr_lambda)

scheduler = cosine_schedule_with_warmup(optimizer, warmup_epochs=5, total_epochs=num_epochs, final_lr_factor=1e-6/3e-4)

#TorchMetrics objects
accuracy = MulticlassAccuracy(num_classes=num_classes, average="macro").to(device)
precision = MulticlassPrecision(num_classes=num_classes, average="macro").to(device)
recall = MulticlassRecall(num_classes=num_classes, average="macro").to(device)
f1 = MulticlassF1Score(num_classes=num_classes, average="macro").to(device)

model.train()
# Training loop
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    total_correct = 0
    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        total_correct += (outputs.argmax(1) == labels).sum().item()

    train_loss = total_loss / len(train_dataset)
    train_acc = total_correct / len(train_dataset)
    print(f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f}")

    # Validation
    model.eval()
    val_loss, val_correct = 0, 0
    accuracy.reset(), precision.reset(), recall.reset(), f1.reset()

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            preds = outputs.argmax(1)
            val_loss += loss.item() * images.size(0)
            val_correct += (preds == labels).sum().item()

            # update torchmetrics
            accuracy.update(preds, labels)
            precision.update(preds, labels)
            recall.update(preds, labels)
            f1.update(preds, labels)

    val_loss /= len(val_dataset)
    val_acc = val_correct / len(val_dataset)
    val_precision = precision.compute().item()
    val_recall = recall.compute().item()
    val_f1 = f1.compute().item()

    print(f"Val Loss: {val_loss:.4f} | Acc: {val_acc:.4f} | "
          f"Precision: {val_precision:.4f} | Recall: {val_recall:.4f} | F1: {val_f1:.4f}")

    scheduler.step()

Epoch 1/50: 100%|██████████| 6302/6302 [18:49<00:00,  5.58it/s]

Train Loss: 8.1050 Acc: 0.0006


Val Loss: 7.7880 | Acc: 0.0012 | Precision: 0.0000 | Recall: 0.0008 | F1: 0.0000


Epoch 2/50: 100%|██████████| 6302/6302 [18:47<00:00,  5.59it/s]

Train Loss: 7.3583 Acc: 0.0038


...

Epoch 25/50: 100%|██████████| 6302/6302 [19:26<00:00,  5.40it/s]

Train Loss: 2.5336 Acc: 0.3215


Val Loss: 3.3311 | Acc: 0.2295 | Precision: 0.2118 | Recall: 0.2171 | F1: 0.1909


Epoch 26/50:  75%|███████▌  | 4754/6302 [14:27<04:42,  5.48it/s]


KeyboardInterrupt: 

In [ ]:
# Save model weights
torch.save(model.state_dict(), "tinyvit_squares.pth")
